# t2i v6.0 · 影子批全版本演进对照（V1 条目制 → r6）

六个批次、同一批 10 实例（29 域等概率抽样）：

| 版本 | 协议要点 |
|---|---|
| V1 条目制 | 条目 = 类别×取值×层级（配额 1:3:6）+ 锚点/可见条件 |
| r2 | 条目制废除，改推理 DAG 文本（`+ / → / [结论] / [不得画]+混淆源`） |
| r3 | 组合类型选择（候选≥3 自然性优先）+ 取景条款限额 ≤2 句 |
| r4 | 输入协议（概念/路径/目标层级）+ 目标层级注入 5:3:2 + 场景迭代 |
| r5 | 考点设计加场景设计（第五部分）+ L3 前提门槛抬升 |
| r6 | 光学去特权位（表行/开关/专项句全清）+ 前提计数机审 + 超层重校验 |

产物：`data/synth_gen_v60_{shadow,r2,r3,r4,r5,r6}/`；抽样 `data/samples_v60_uniform.jsonl`。


In [ ]:
# ---- 加载（全部批次：V1 条目制 + r2~r6 DAG 制） ----
import json as _json
import html as _html
import re as _re
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

T2I_ROOT = Path.cwd() if (Path.cwd() / 'data' / 'samples_v60_uniform.jsonl').exists() else Path('..')
if not (T2I_ROOT / 'data' / 'samples_v60_uniform.jsonl').exists():
    T2I_ROOT = Path.cwd() / 'benchmark' / 't2i'
_DATA = T2I_ROOT / 'data'

_BATCHES = [          # (标签, 目录, 批次说明)
    ('V1 条目制', 'synth_gen_v60_shadow', '条目=类别×取值×层级，配额 1:3:6'),
    ('r2', 'synth_gen_v60_r2', 'DAG 制首版'),
    ('r3', 'synth_gen_v60_r3', '组合选择+取景限额'),
    ('r4', 'synth_gen_v60_r4', '目标层级注入 5:3:2'),
    ('r5', 'synth_gen_v60_r5', '场景设计+L3前提≥6*'),
    ('r6', 'synth_gen_v60_r6', '光学去特权位+前提计数机审'),
]
_TGT = {'60001': 'L3', '60002': 'L2', '60003': 'L3', '60004': 'L2', '60005': 'L3',
        '60006': 'L2', '60007': 'L3', '60008': 'L3', '60009': 'L2', '60010': 'L1'}

def _load(fp):
    return [_json.loads(l) for l in fp.read_text(encoding='utf-8').splitlines() if l.strip()]
_smp = {r['instance']: r for r in _load(_DATA / 'samples_v60_uniform.jsonl')}
_qs = {}                     # {tag: {sid: q}}
for tag, d, _desc in _BATCHES:
    fp = _DATA / d / 'questions_v60_gpt-5.6-sol.jsonl'
    if fp.exists():
        _qs[tag] = {q['_job_sample']: q for q in _load(fp)}
_sids = sorted({sid for m in _qs.values() for sid in m})
print(' | '.join(f'{t}: {len(m)} 题' for t, m in _qs.items()))
print('共同实例', len(_sids), '个；目标注入（r4 起）：', _TGT)

def _opt(q):
    t = (q.get('reasoning') or '') + (q.get('gen_prompt') or '')
    return bool(_re.search(r'镜像|倒影|镜面|反射|水面|水洼|平面镜|镜中|镜墙', t))

def _bo(q):
    return len(_re.findall(r'无遮挡|细节清晰|清晰可辨|清晰可见', q.get('gen_prompt') or ''))


In [ ]:
# ---- 批次演进总览（指标 × 版本） ----
_rows = []
for tag, d, desc in _BATCHES:
    m = _qs.get(tag)
    if not m:
        continue
    dag = all('reasoning' in (q or {}) for q in m.values())
    if dag:
        lv = Counter(q.get('level') for q in m.values())
    else:
        lv = Counter(c.get('level') for q in m.values()
                     for c in q.get('check_items') or [])
    pres = [q['reasoning'].count('（前提）') for q in m.values()] if dag else [0]
    cons = ([q['reasoning'].count('[结论]') for q in m.values()] if dag
            else [len(q.get('check_items') or []) for q in m.values()])
    _rows.append({
        '版本': tag, '说明': desc,
        'L1/L2/L3': f"{lv.get('L1',0)}/{lv.get('L2',0)}/{lv.get('L3',0)}"
                     + ('（条目）' if not dag else ''),
        '光学题数': f"{sum(1 for q in m.values() if _opt(q))}/{len(m)}",
        '前提中位': sorted(pres)[len(pres)//2] if dag else '-',
        '结论中位': sorted(cons)[len(cons)//2],
        '套话词频': sum(_bo(q) for q in m.values()),
    })
_ev = pd.DataFrame(_rows)
with pd.option_context('display.max_colwidth', 40):
    display(_ev)


In [ ]:
# ---- 分布对照（各版本层级 / 光学占比） ----
plt.rcParams['font.family'] = ['Noto Sans CJK SC', 'DejaVu Sans']  # CJK 优先，避免中文豆腐块
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none',
                     'text.color': '#cccccc', 'axes.edgecolor': '#555555',
                     'xtick.color': '#999999', 'ytick.color': '#999999',
                     'axes.labelcolor': '#cccccc'})
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
tags = [t for t, _, _ in _BATCHES if t in _qs]
ax = axes[0]
w = 0.8 / len(tags)
for k, tag in enumerate(tags):
    lv = Counter(q.get('level') for q in _qs[tag].values())
    ax.bar([x + (k - len(tags) / 2 + .5) * w for x in (1, 2, 3)],
           [lv.get(f'L{i}', 0) for i in (1, 2, 3)], width=w, label=tag)
ax.set_xticks([1, 2, 3]); ax.set_xticklabels(['L1', 'L2', 'L3']); ax.legend(fontsize=8)
ax.set_title('层级分布（各版本）')
ax = axes[1]
ax.bar(tags, [sum(1 for q in _qs[t].values() if _opt(q)) for t in tags], color='#c0392b')
ax.set_title('光学媒介题数（粗匹配）')
ax.tick_params(axis='x', labelsize=8, rotation=20)
plt.tight_layout(); plt.show()


In [ ]:
# ---- 机审复核（DAG 批次；audit_v60 零 LLM 调用） ----
import sys
sys.path.insert(0, str(T2I_ROOT.parent))
from t2i.eval_synthesize import audit_v60
for tag, _, _ in _BATCHES:
    m = _qs.get(tag)
    if not m or not all('reasoning' in q for q in m.values()):
        continue
    n_warn = 0
    for sid, q in sorted(m.items()):
        warns = audit_v60(q, None)
        if warns:
            n_warn += 1
            print(f"[{tag}] [{sid}] {'; '.join(warns)}")
    print(f'{tag}: ' + ('机审零告警' if n_warn == 0 else f'告警 {n_warn} 题'
                         + '（注：r6 前的批次按现行门槛回溯判）'))


In [ ]:
# ---- 全版本逐题对照卡片（SHOW = 'all' 或 sid 列表；VER = 显示的版本标签列表） ----
SHOW = 'all'
VER = [t for t, _, _ in _BATCHES]          # 可裁剪如 ['V1 条目制', 'r6']
_LVC = {'L1': '#7f8c8d', 'L2': '#2471a3', 'L3': '#c0392b'}
_ACCENTS = ['#7f8c8d', '#b8860b', '#8a6d3b', '#2471a3', '#6a5acd', '#2a7a4b']

def _img_html(sid):
    entries = []
    any_q = next((m[sid] for m in _qs.values() if sid in m), {})
    ip = T2I_ROOT / 'data' / str(any_q.get('_sample_image') or '')
    if ip.exists():
        entries.append((ip, '样本原图'))
    gp = T2I_ROOT / 'data' / 'eval_v60_shadow' / 'bagel' / 'imgs' / f'{sid}.png'
    if gp.exists():
        entries.append((gp, 'Bagel 生成图（V1 题面）'))
    if not entries:
        return ''
    import base64
    parts = []
    for pth, cap in entries:
        b64 = base64.b64encode(pth.read_bytes()).decode('ascii')
        mime = 'image/png' if pth.suffix == '.png' else 'image/jpeg'
        parts.append(f'<div style="display:inline-block;text-align:center;margin-left:8px">'
                     f'<img src="data:{mime};base64,{b64}" style="width:170px;border-radius:6px">'
                     f'<div style="color:#999;font-size:11px">{cap}</div></div>')
    return f'<div style="float:right;margin:0 0 8px 10px">{"".join(parts)}</div>'

def _card(q, tag, accent):
    if not q:
        return f'<div style="color:#999">{tag} 缺题</div>'
    # if 'reasoning' in q:                     # DAG 版
    #     lvl = q.get('level', '?')
    #     rs = _html.escape(str(q.get('reasoning') or '（无）')).replace(chr(10), '<br>')
    #     body = ('<div style="background:rgba(128,128,128,0.08);border:1px solid '
    #             'rgba(128,128,128,0.30);border-radius:4px;padding:8px 10px;'
    #             f'font-size:12.5px;line-height:1.7"><b>推理 DAG</b><br>{rs}</div>')
    #     lr = (f'<div style="color:#aaa;font-size:12px;margin-bottom:6px">'
    #           f'<b>层级依据</b>：{_html.escape(str(q.get("level_reason", "") or "-"))}</div>')
    # else:                                     # V1 条目制
    #     lvl = f"{len(q.get('check_items') or [])} 条"
    #     rows = []
    #     for i, c in enumerate(q.get('check_items') or [], 1):
    #         chain = ' → '.join(_html.escape(str(x)) for x in (c.get('chain') or [])) or '（题面直述）'
    #         rows.append(f'<tr><td style="padding:2px 6px;vertical-align:top">'
    #                     f'<span style="background:{_LVC.get(c.get("level"), "#999")};color:#fff;'
    #                     f'padding:0 5px;border-radius:3px;font-size:10px">{c.get("level","?")}</span></td>'
    #                     f'<td style="vertical-align:top"><b>{_html.escape(str(c.get("category","")))}</b> · '
    #                     f'{_html.escape(str(c.get("value","")))}<br>'
    #                     f'<span style="color:#999;font-size:12px">链: {chain} ｜ 锚点: '
    #                     f'{_html.escape(str(c.get("anchor","")))}</span></td></tr>')
    #     body = (f'<table style="border-collapse:collapse;width:100%">{"".join(rows)}</table>')
    #     lr = ''
    notes = (q.get('notes') or '').strip()
    notes_html = ('<div style="background:rgba(200,170,60,0.15);padding:5px 10px;'
                  f'border-radius:4px;margin-top:6px">📝 {_html.escape(notes)}</div>') if notes else ''
    return (
        f'<div style="border:1px solid rgba(128,128,128,0.35);border-left:4px solid {accent};'
        'border-radius:6px;padding:10px 12px;margin:8px 0;font-size:13px">'
        f'<div style="margin-bottom:4px"><span style="color:{accent};font-weight:bold">{tag}</span>'
        f'<span style="background:{_LVC.get(q.get("level"), "#666")};color:#fff;padding:1px 8px;'
        f'border-radius:3px;margin-left:6px;font-size:12px">{lvl}</span>'
        + (f'<span style="color:#888;font-size:11px;margin-left:6px">目标 {_TGT.get(sid, "?")}</span>'
           if tag not in ('V1 条目制', 'r2', 'r3') else '')
        + '</div>'
        '<div style="background:rgba(128,128,128,0.12);padding:6px 10px;border-radius:4px;'
        f'margin-bottom:4px"><b>题面</b>：{_html.escape(q.get("gen_prompt", ""))}</div>'
        f'{lr}{body}{notes_html}</div>')

sels = _sids if SHOW == 'all' else [s for s in _sids if s in SHOW]
for sid in sels:
    any_q = next((m[sid] for m in _qs.values() if sid in m), {})
    concept = any_q.get('_query_label', '?')
    s = _smp.get(concept, {})
    paths = '<br>'.join(_html.escape(pp) for pp in (s.get('mount_paths') or []))
    head = ('<div style="border:1px solid rgba(128,128,128,0.5);border-radius:8px;'
            'padding:12px 14px;margin:18px 0;overflow:auto">'
            f'<div style="font-size:15px;margin-bottom:2px"><b>[{sid}] {_html.escape(concept)}</b> '
            f'<span style="color:#888;font-size:12px">{_html.escape(str(s.get("l1", "?")))}'
            f' / {_html.escape(str(s.get("l2", "?")))}</span></div>'
            f'<div style="color:#999;font-size:11px;margin-bottom:6px">{paths}</div>'
            f'{_img_html(sid)}')
    cards = ''.join(_card(_qs.get(t, {}).get(sid), t, a)
                    for t, a in zip([x for x in VER if x in _qs], _ACCENTS))
    display(HTML(head + cards + '</div>'))


In [ ]:
# ---- 题面清单（全版本对照） ----
for sid in _sids:
    any_q = next((m[sid] for m in _qs.values() if sid in m), {})
    print(f"[{sid}] {any_q.get('_query_label')}")
    for tag, _, _ in _BATCHES:
        q = _qs.get(tag, {}).get(sid)
        if q:
            print(f"  {tag} ({q.get('level')}): {q.get('gen_prompt')}")
    print()
